In [1]:
import pandas as pd
import numpy as np

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "object_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../../data/raw/srodmiescie_raw_object_id.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
).dropna(subset="ground_floor_function")

In [3]:
data_gdynia["date_of_checking"] = pd.to_datetime(
    data_gdynia["date_of_checking"], format="%d.%m.%Y"
)

In [4]:
data_clean = data_gdynia.assign(
    ground_floor_function=lambda d: (
        d["ground_floor_function"].str.replace("_", " ").str.lower()
    )
)

In [5]:
street_fixes = {
    "Hryniewickiego": "Antoniego Hryniewickiego",
    "Wendy": "Tadeusza Wendy",
    "Plac Kaszubski - Jana z Kolna": "Jana z Kolna",
}

data_clean["addr_street"] = data_clean["addr_street"].replace(street_fixes)

data_clean["addr_number"] = data_clean["addr_number"].str.strip().str.upper()

In [ ]:
style_fixes = {
    "nowożytnośc": "współczesność",
    "współczsnośc": "współczesność",
    "wspóółczesność": "współczesność",
    "wspóczesność": "współczesność",
    "współczesnosc": "współczesność",
    "traycjonalizm": "tradycjonalizm",
    "wiejski": "tradycjonalizm",
    "art. deco": "modernizm",
    "artdeco": "modernizm",
    "wertykalizm": "modernizm",
    "nowożytność": "współczesność",
}

data_clean["arch_style"] = (
    data_clean["arch_style"].str.lower().str.strip().replace(style_fixes)
)

In [ ]:
data_object_id = data_clean.loc[data_clean["object_id"].notna()].reset_index()
data_remainig = data_clean.loc[data_clean["object_id"].isna()].reset_index()

In [ ]:
assert (
    data_object_id.groupby("object_id")["arch_style"].nunique(dropna=False) == 1
).all()
assert (
    data_object_id.groupby("object_id")["build_year_est"].nunique(dropna=False) == 1
).all()

In [ ]:
def connect_func(s):
    func = s.str.split(",").explode().str.strip().str.lower().dropna().unique()
    return ",".join(func) if len(func) > 0 else pd.Na


data_merged = (
    data_object_id.sort_values("number_of_storeys", ascending=False)
    .groupby("object_id", sort=False)
    .agg(
        id=("id", "min"),
        osm_id=("osm_id", ",".join),
        addr_street=("addr_street", "first"),
        addr_number=("addr_number", "first"),
        ground_floor_group=("ground_floor_group", "first"),
        ground_floor_function=("ground_floor_function", connect_func),
        upper_floor_group=("upper_floor_group", "first"),
        upper_floor_function=("upper_floor_function", "first"),
        number_of_storeys=("number_of_storeys", "max"),
        is_mixed_use=("is_mixed_use", "max"),
        build_year_est=("build_year_est", "first"),
        arch_style=("arch_style", "first"),
        condition=("condition", "first"),
        max_height=("max_height", "max"),
        is_heritage=("is_heritage", "first"),
        source=("source", "first"),
        confidence=("confidence", "min"),
        date_of_checking=("date_of_checking", "max"),
    )
    .reset_index()
)
data_single = data_remainig.assign(object_id=lambda d: d["id"].astype(str))[
    data_merged.columns
]

data_objects = (
    pd.concat([data_merged, data_single], ignore_index=True)
    .set_index("id")
    .sort_index()
    .drop(columns=["object_id"])
    .assign(upper_floor_function=lambda d: d["upper_floor_function"].str.lower())
)

In [10]:
data_objects_long = (
    data_objects.assign(
        ground_floor_function=lambda d: d["ground_floor_function"].str.split(",")
    )
    .explode("ground_floor_function")
    .assign(ground_floor_function=lambda d: d["ground_floor_function"].str.strip())
)
data_objects_long = data_objects_long.loc[:, ["addr_street", "ground_floor_function"]]

In [11]:
category_map = pd.read_csv(
    "../../data/reference/category_map.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)

In [ ]:
data_long_categories = pd.merge(
    data_objects_long,
    category_map,
    left_on="ground_floor_function",
    right_index=True,
    indicator=True,
    how="outer",
    validate="many_to_one",
)

data_long_categories = data_long_categories.loc[
    :, ["addr_street", "function_clean", "category"]
]
data_objects = data_objects.assign(
    number_of_functions=data_long_categories.groupby("id").agg("size")
)

In [ ]:
assert data_objects["number_of_functions"].notna().all()
assert data_objects["number_of_functions"].sum() == len(data_long_categories)

In [ ]:
data_long_categories.to_csv(
    "../../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    encoding="utf-8",
)
data_objects.to_csv(
    "../../data/processed/fixed/srodmiescie.csv", sep=";", encoding="utf-8"
)